<a href="https://colab.research.google.com/github/Shiya30-sys/IntentGuard/blob/main/RAG_PIPELINE_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip -q install pinecone sentence-transformers langchain langchain-openai pyyaml

## upserting

In [10]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPEN_ROUTER_API_KEY'] = userdata.get('OPEN_ROUTER_API_KEY')
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')

In [19]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [24]:
data_dir = '/content/drive/My Drive/main_drive/iit-jammu/batch_1/data'
os.listdir(data_dir)

[]

### Step 1: Connect to the vector DB

In [26]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone()
# I want to meet vector-hybrid
pc.list_indexes()

IndexList([])

### Step 2: Check if your index exists, if not then create it

In [27]:
index_name = 'vector1'
pc.list_indexes()

IndexList([])

In [28]:
if index_name not in [idx.name for idx in pc.list_indexes()]:
  print(f"Creating index {index_name}")
  pc.create_index(
      name=index_name,
      dimension=384, # Must match paraphrase-MiniLM-L6-v2
      metric='cosine',
      spec=ServerlessSpec(cloud='aws', region='us-east-1')
  )
else:
  print("Index already exists")

Creating index vector1


In [30]:
index = pc.Index(index_name)

In [31]:
%pip install -q chromadb sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


#### For CHROMA DB

In [32]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

data_dir = "/content/drive/My Drive/main_drive/iit-jammu/batch_1/data"

chroma_db_dir = '/content/drive/My Drive/main_drive/iit-jammu/batch_1'

index_name = 'vector1'



# Create Embedding function
embedding_function = SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-MiniLM-L6-v2",
    device="cpu",
)

cc = chromadb.PersistentClient(
    path=chroma_db_dir
)


# Get a list of already existing vector DBs inside chroma db
collection_names = [
    c if isinstance(c, str) else c.name
    for c in cc.list_collections()
]



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [33]:
# If vectorDb already doesn't exist, then create it.
if index_name not in collection_names:

    print(f"Creating collection {index_name}")

    collection = cc.create_collection(
        name=index_name,
        metadata={
            "hnsw:space": "cosine"
        },
        embedding_function=embedding_function,
    )

else:

    print("Collection already exists")

    collection = cc.get_collection(
        name=index_name,
        embedding_function=embedding_function,
    )


Creating collection vector1


In [34]:
print("Collection:", collection.name)
print("Records:", collection.count())

Collection: vector1
Records: 0


## Step 3: Define your documents

In [35]:
## DOC 1 -
# lskdsldjsd;ljsd;ljsd;ljds;ldsj
# ;sj;sdljd;ljds;ljsd;lsdj;ldjsd
# ;s;sldjsd;ljsd;ljsd;lj;ldsj
# ;lshds;ljd;h;lsgklfjha;lsfhdfs
# ldfah;lfdh;laskfdh;dlsahfsd
# ;lskhadaflkshflhs;ldh
# ;lsdhlkfdsh;ldsh;dslhds;flkh





  ## You need to divide documents inot chunks and give each chunk an ID
docs = [
    {"id": "doc1", "text": "Pandas is a Python library for data analysis."},
    {"id": "doc2", "text": "Pinecone is a vector database for semantic search."},
    {"id": "doc3", "text": "Spark enables distributed data processing."},
]

### Step 4: Create your Embeddings

In [36]:
## Step 4: Create your Embeddings
[doc["text"] for doc in docs]

['Pandas is a Python library for data analysis.',
 'Pinecone is a vector database for semantic search.',
 'Spark enables distributed data processing.']

In [37]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")
doc_list = [doc["text"] for doc in docs]
embeddings = model.encode(doc_list)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [38]:
[embedding[0:5] for embedding in embeddings]

[array([-0.4884024 , -0.7727132 , -0.17936738,  0.2750812 ,  0.10556224],
       dtype=float32),
 array([-0.34000304, -0.20315416, -0.0631818 ,  0.02528497,  0.07103913],
       dtype=float32),
 array([-0.60318345, -0.419841  , -0.5404588 , -0.26298213,  0.09807338],
       dtype=float32)]

### Step 5: Organize the documents and Embeddings together

In [39]:
# THis is how you need to organize your vector with id, values, and metadata
# All additional information like text, keywords, headings, etc go inside metadata
{'id': 'doc1',
 'values': [-0.48840246, -0.7727129, -0.17936754, 0.27508107, 0.10556235],
 'metadata': {'text': 'Pandas is a Python library for data analysis.'}}

{'id': 'doc1',
 'values': [-0.48840246, -0.7727129, -0.17936754, 0.27508107, 0.10556235],
 'metadata': {'text': 'Pandas is a Python library for data analysis.'}}

In [41]:
# How you package the vector and the docs together
embeddings_short =  [embedding[0:5] for embedding in embeddings]

In [42]:
[{'id': d['id'],
 'values': emb.tolist(),
 'metadata': {'text': d['text']}}
for d, emb in zip(docs, embeddings_short)]

[{'id': 'doc1',
  'values': [-0.488402396440506,
   -0.7727131843566895,
   -0.17936737835407257,
   0.27508118748664856,
   0.1055622398853302],
  'metadata': {'text': 'Pandas is a Python library for data analysis.'}},
 {'id': 'doc2',
  'values': [-0.34000304341316223,
   -0.20315416157245636,
   -0.0631817951798439,
   0.025284968316555023,
   0.07103913277387619],
  'metadata': {'text': 'Pinecone is a vector database for semantic search.'}},
 {'id': 'doc3',
  'values': [-0.6031834483146667,
   -0.4198409914970398,
   -0.5404587984085083,
   -0.2629821300506592,
   0.09807337820529938],
  'metadata': {'text': 'Spark enables distributed data processing.'}}]

In [43]:
# We're now packaging the docs and embeddings together and then organizing them according our defined structure above
# DOING IT WITH REAL FULL EMBEDDINGS
vectors = [{'id': d['id'],
 'values': emb.tolist(),
 'metadata': {'text': d['text']}}
for d, emb in zip(docs, embeddings)]


#### Chroma DB

In [44]:
from sentence_transformers import SentenceTransformer

docs = [
    {"id": "doc1", "text": "Pandas is a Python library for data analysis."},
    {"id": "doc2", "text": "Pinecone is a vector database for semantic search."},
    {"id": "doc3", "text": "Spark enables distributed data processing."},
]

model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

ids = [doc["id"] for doc in docs]
documents = [doc["text"] for doc in docs]

embeddings = model.encode(documents)

metadatas = [
    {"source": "training_docs"}
    for _ in docs
]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Step 6: Upsert to Vector DB

In [45]:
index.upsert(vectors = vectors)

UpsertResponse(upserted_count=3)

#### Chroma DB

In [46]:
collection.upsert(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=documents,
    metadatas=metadatas,
)

print(f"Inserted {collection.count()} documents")

Inserted 3 documents
